In [ ]:
# ==============================
# PLANT DISEASE DETECTION USING CNN
# ==============================

# Install libraries (Run only first time)
# !pip install tensorflow matplotlib

# ==============================
# IMPORT LIBRARIES
# ==============================

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image

import matplotlib.pyplot as plt
import numpy as np

# ==============================
# DATASET PATH
# ==============================

dataset_path = "PlantVillage"

# ==============================
# IMAGE PREPROCESSING
# ==============================

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    dataset_path,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

# ==============================
# BUILD CNN MODEL
# ==============================

model = Sequential()

# 1st CNN Layer
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
model.add(MaxPooling2D(pool_size=(2,2)))

# 2nd CNN Layer
model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(pool_size=(2,2)))

# Flatten Layer
model.add(Flatten())

# Dense Layer
model.add(Dense(128, activation='relu'))

# Output Layer
model.add(Dense(train_data.num_classes, activation='softmax'))

# ==============================
# COMPILE MODEL
# ==============================

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ==============================
# TRAIN MODEL
# ==============================

history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5
)

# ==============================
# SAVE MODEL
# ==============================

model.save("plant_disease_model.h5")

print("Model Saved Successfully!")

# ==============================
# ACCURACY GRAPH
# ==============================

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend(['Training Accuracy', 'Validation Accuracy'])

plt.show()

# ==============================
# TEST NEW IMAGE
# ==============================

# Give image path here
img_path = "test.jpg"

# Load image
img = image.load_img(img_path, target_size=(128,128))

# Convert image to array
img_array = image.img_to_array(img)

# Expand dimensions
img_array = np.expand_dims(img_array, axis=0)

# Normalize image
img_array = img_array / 255.0

# Prediction
prediction = model.predict(img_array)

# Get class labels
class_names = list(train_data.class_indices.keys())

# Predicted class
result = class_names[np.argmax(prediction)]

print("Predicted Disease :", result)